# Hybrid Quantum-Classical Transformer (HQCT)

**Paper:** *Hybrid Quantum-Classical Attention Mechanisms for Parameter-Efficient Transformers in Natural Language Processing*  
**Author:** Amisha Singh, The NorthCap University

This notebook implements the full experimental pipeline:

| Variant | Description |
|---------|-------------|
| **HQCT-K** | Classical Q/K/V + **quantum kernel** similarity $\kappa(q,k)=\|\langle\psi(q)\|\psi(k)\rangle\|^2$ |
| **HQCT-P** | Quantum circuits replace $W_Q$, $W_K$; value stays classical |
| **Classical** | Matched Transformer baseline |

**Default config (paper §4.2):** 4 layers · $d=256$ · 4 heads · 6 qubits · depth 3


## 1. Install dependencies

In [1]:
# Run once
!pip install -q torch pennylane pennylane-lightning transformers datasets scikit-learn tqdm matplotlib numpy


'pip' is not recognized as an internal or external command,
operable program or batch file.


## 2. Imports & configuration

In [2]:
import math
import copy
import time
from typing import Optional, Tuple, List, Dict, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset, Subset
from tqdm.auto import tqdm

import pennylane as qml
import matplotlib.pyplot as plt

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


c:\Users\amisa\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


## 3. Quantum circuits (hardware-efficient ansatz)

In [3]:
def make_device(n_qubits: int, noise: float = 0.0):
    """PennyLane device – Lightning for speed, default.qubit as fallback."""
    if noise > 0.0:
        try:
            from qiskit_aer.noise import NoiseModel, depolarizing_error
            nm = NoiseModel()
            nm.add_all_qubit_quantum_error(depolarizing_error(noise, 1), ["rx", "ry", "rz"])
            nm.add_all_qubit_quantum_error(depolarizing_error(noise, 2), ["cx"])
            return qml.device("qiskit.aer", wires=n_qubits, backend="aer_simulator_statevector", noise_model=nm)
        except Exception:
            pass
    try:
        return qml.device("lightning.qubit", wires=n_qubits)
    except Exception:
        return qml.device("default.qubit", wires=n_qubits)


def hardware_efficient_ansatz(features, weights, n_qubits: int, n_layers: int):
    """Angle embedding + L layers of (Rx, Ry, Rz) + nearest-neighbour CNOTs."""
    for i in range(n_qubits):
        qml.RX(features[i], wires=i)
        qml.RY(features[i], wires=i)
    for layer in range(n_layers):
        for i in range(n_qubits):
            qml.RX(weights[layer, i, 0], wires=i)
            qml.RY(weights[layer, i, 1], wires=i)
            qml.RZ(weights[layer, i, 2], wires=i)
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])


class QuantumKernel(nn.Module):
    """κ(q, k) = |⟨ψ(q)|ψ(k)⟩|²  – used by HQCT-K."""

    def __init__(self, n_qubits=6, n_layers=3, noise=0.0):
        super().__init__()
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        self.weights = nn.Parameter(0.01 * torch.randn(n_layers, n_qubits, 3))
        self._dev = make_device(n_qubits, noise)
        self._circuit = self._build()

    def _build(self):
        @qml.qnode(self._dev, interface="torch", diff_method="parameter-shift")
        def circuit(fq, fk, weights):
            hardware_efficient_ansatz(fq, weights, self.n_qubits, self.n_layers)
            qml.adjoint(hardware_efficient_ansatz)(fk, weights, self.n_qubits, self.n_layers)
            return qml.probs(wires=range(self.n_qubits))
        return circuit

    def forward(self, q: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
        # q, k: (B, H, N, n_qubits) or (N, n_qubits)
        orig = q.shape
        if q.dim() == 4:
            B, H, N, D = q.shape
            q, k = q.reshape(B * H, N, D), k.reshape(B * H, N, D)
            flat = True
        else:
            flat, N = False, q.shape[0]
        bs = q.shape[0]
        kappa = torch.zeros(bs, N, N, device=q.device, dtype=q.dtype)
        for b in range(bs):
            for i in range(N):
                for j in range(N):
                    kappa[b, i, j] = self._circuit(q[b, i], k[b, j], self.weights)[0]
        return kappa.reshape(B, H, N, N) if flat else kappa


class QuantumProjection(nn.Module):
    """Maps classical vector → quantum state → Pauli-Z expectations (HQCT-P)."""

    def __init__(self, in_dim, n_qubits=6, n_layers=3, noise=0.0, correlators=False):
        super().__init__()
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        self.correlators = correlators
        self.pre_map = nn.Linear(in_dim, n_qubits)
        self.weights = nn.Parameter(0.01 * torch.randn(n_layers, n_qubits, 3))
        self.out_dim = n_qubits + (n_qubits - 1 if correlators else 0)
        self._dev = make_device(n_qubits, noise)
        self._circuit = self._build()

    def _build(self):
        @qml.qnode(self._dev, interface="torch", diff_method="parameter-shift")
        def circuit(features, weights):
            hardware_efficient_ansatz(features, weights, self.n_qubits, self.n_layers)
            expvals = [qml.expval(qml.PauliZ(i)) for i in range(self.n_qubits)]
            if self.correlators:
                for i in range(self.n_qubits - 1):
                    expvals.append(qml.expval(qml.PauliZ(i) @ qml.PauliZ(i + 1)))
            return expvals
        return circuit

    def forward(self, x):
        feat = self.pre_map(x)
        shape = feat.shape[:-1]
        feat = feat.reshape(-1, self.n_qubits)
        outs = [torch.stack(self._circuit(feat[i], self.weights)) for i in range(feat.shape[0])]
        return torch.stack(outs).reshape(*shape, self.out_dim)

print("Quantum modules defined ✓")


Quantum modules defined ✓


## 4. Attention modules & HQCT model

In [4]:
class ClassicalMHA(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.n_heads, self.d_k = n_heads, d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask=None, return_attn=False):
        B, N, _ = x.shape
        q = self.W_q(x).view(B, N, self.n_heads, self.d_k).transpose(1, 2)
        k = self.W_k(x).view(B, N, self.n_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(B, N, self.n_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attn = self.drop(F.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, N, -1)
        out = self.W_o(out)
        return (out, attn) if return_attn else (out, None)


class QuantumKernelAttention(nn.Module):
    """HQCT-K"""
    def __init__(self, d_model, n_heads, n_qubits=6, n_layers=3, dropout=0.1, noise=0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.q_map = nn.Linear(self.d_k, n_qubits)
        self.k_map = nn.Linear(self.d_k, n_qubits)
        self.qkernel = QuantumKernel(n_qubits, n_layers, noise)
        self.drop = nn.Dropout(dropout)
        self.scale = math.sqrt(n_qubits)

    def forward(self, x, mask=None, return_attn=False):
        B, N, _ = x.shape
        q = self.W_q(x).view(B, N, self.n_heads, self.d_k).transpose(1, 2)
        k = self.W_k(x).view(B, N, self.n_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(B, N, self.n_heads, self.d_k).transpose(1, 2)
        kappa = self.qkernel(self.q_map(q), self.k_map(k))
        scores = kappa / self.scale
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attn = self.drop(F.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, N, -1)
        out = self.W_o(out)
        return (out, attn) if return_attn else (out, None)


class QuantumProjectionAttention(nn.Module):
    """HQCT-P"""
    def __init__(self, d_model, n_heads, n_qubits=6, n_layers=3, dropout=0.1, noise=0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.q_proj = QuantumProjection(d_model, n_qubits, n_layers, noise)
        self.k_proj = QuantumProjection(d_model, n_qubits, n_layers, noise)
        self.q_post = nn.Linear(self.q_proj.out_dim, d_model)
        self.k_post = nn.Linear(self.k_proj.out_dim, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask=None, return_attn=False):
        B, N, _ = x.shape
        q = self.q_post(self.q_proj(x)).view(B, N, self.n_heads, self.d_k).transpose(1, 2)
        k = self.k_post(self.k_proj(x)).view(B, N, self.n_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(B, N, self.n_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attn = self.drop(F.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, N, -1)
        out = self.W_o(out)
        return (out, attn) if return_attn else (out, None)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1,
                 attn_type="classical", n_qubits=6, q_layers=3, noise=0.0):
        super().__init__()
        if attn_type == "classical":
            self.attn = ClassicalMHA(d_model, n_heads, dropout)
        elif attn_type == "hqct-k":
            self.attn = QuantumKernelAttention(d_model, n_heads, n_qubits, q_layers, dropout, noise)
        elif attn_type == "hqct-p":
            self.attn = QuantumProjectionAttention(d_model, n_heads, n_qubits, q_layers, dropout, noise)
        else:
            raise ValueError(attn_type)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.n1 = nn.LayerNorm(d_model)
        self.n2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask=None, return_attn=False):
        a, w = self.attn(self.n1(x), mask, return_attn)
        x = x + self.drop(a)
        x = x + self.ff(self.n2(x))
        return x, w


class HQCT(nn.Module):
    """Hybrid Quantum-Classical Transformer (paper §3)."""
    def __init__(self, vocab_size, num_classes, d_model=256, n_layers=4, n_heads=4,
                 d_ff=512, max_seq_len=256, dropout=0.1, attn_type="hqct-k",
                 n_qubits=6, q_layers=3, noise=0.0, pad_idx=0):
        super().__init__()
        self.pad_idx = pad_idx
        self.tok = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos = nn.Embedding(max_seq_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout, attn_type, n_qubits, q_layers, noise)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.cls = nn.Linear(d_model, num_classes)
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, input_ids, attention_mask=None, return_attn=False):
        B, N = input_ids.shape
        pos = torch.arange(N, device=input_ids.device).unsqueeze(0).expand(B, -1)
        x = self.drop(self.tok(input_ids) + self.pos(pos))
        if attention_mask is None:
            attention_mask = (input_ids != self.pad_idx).long()
        mask = attention_mask.unsqueeze(1).unsqueeze(2)
        attns = []
        for layer in self.layers:
            x, a = layer(x, mask, return_attn)
            if return_attn:
                attns.append(a)
        x = self.norm(x)
        m = attention_mask.unsqueeze(-1).float()
        pooled = (x * m).sum(1) / m.sum(1).clamp(min=1e-9)
        logits = self.cls(pooled)
        return (logits, attns) if return_attn else logits

print("HQCT model defined ✓")


HQCT model defined ✓


## 5. Data loading (AG News, IMDB, SST-2, MultiNLI)

In [5]:
from datasets import load_dataset
from transformers import AutoTokenizer

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts, self.labels = texts, labels
        self.tok, self.max_length = tokenizer, max_length
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        enc = self.tok(self.texts[i], truncation=True, padding="max_length",
                       max_length=self.max_length, return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[i], dtype=torch.long),
        }


def load_benchmark(name, max_length=None, val_ratio=0.05, seed=42):
    tok = AutoTokenizer.from_pretrained("bert-base-uncased")
    name = name.lower()
    if name == "ag_news":
        ds = load_dataset("ag_news")
        tr_t, tr_y = ds["train"]["text"], ds["train"]["label"]
        te_t, te_y = ds["test"]["text"], ds["test"]["label"]
        n_cls, max_length = 4, max_length or 128
    elif name == "imdb":
        ds = load_dataset("imdb")
        tr_t, tr_y = ds["train"]["text"], ds["train"]["label"]
        te_t, te_y = ds["test"]["text"], ds["test"]["label"]
        n_cls, max_length = 2, max_length or 256
    elif name == "sst2":
        ds = load_dataset("glue", "sst2")
        tr_t, tr_y = ds["train"]["sentence"], ds["train"]["label"]
        te_t, te_y = ds["validation"]["sentence"], ds["validation"]["label"]
        n_cls, max_length = 2, max_length or 128
    elif name == "multinli":
        ds = load_dataset("multi_nli")
        tr_t = [p + " [SEP] " + h for p, h in zip(ds["train"]["premise"], ds["train"]["hypothesis"])]
        tr_y = ds["train"]["label"]
        idx = [i for i, y in enumerate(tr_y) if y != -1]
        tr_t, tr_y = [tr_t[i] for i in idx], [tr_y[i] for i in idx]
        val = ds["validation_matched"]
        te_t = [p + " [SEP] " + h for p, h in zip(val["premise"], val["hypothesis"])]
        te_y = val["label"]
        n_cls, max_length = 3, max_length or 128
    else:
        raise ValueError(name)

    rng = np.random.RandomState(seed)
    n = len(tr_t)
    perm = rng.permutation(n)
    n_val = max(1, int(n * val_ratio))
    val_idx, tr_idx = perm[:n_val], perm[n_val:]
    train_ds = TextDataset([tr_t[i] for i in tr_idx], [tr_y[i] for i in tr_idx], tok, max_length)
    val_ds   = TextDataset([tr_t[i] for i in val_idx], [tr_y[i] for i in val_idx], tok, max_length)
    test_ds  = TextDataset(te_t, te_y, tok, max_length)
    return train_ds, val_ds, test_ds, n_cls, tok.vocab_size, max_length


def make_few_shot(dataset, n_classes, k=16, seed=42):
    rng = np.random.RandomState(seed)
    labels = [dataset[i]["labels"].item() for i in range(len(dataset))]
    indices = []
    for c in range(n_classes):
        cls = [i for i, y in enumerate(labels) if y == c]
        indices.extend(rng.choice(cls, size=min(k, len(cls)), replace=False).tolist())
    return Subset(dataset, indices)

print("Data utilities ready ✓")


Data utilities ready ✓


## 6. Training & evaluation utilities

In [6]:
def separate_params(model):
    classical, quantum = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "weights" in name and any(k in name for k in ("qkernel", "q_proj", "k_proj", "quantum")):
            quantum.append(p)
        else:
            classical.append(p)
    return classical, quantum


def train_epoch(model, loader, opt, criterion, device):
    model.train()
    total, n = 0.0, 0
    for batch in tqdm(loader, desc="train", leave=False):
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        y = batch["labels"].to(device)
        opt.zero_grad()
        loss = criterion(model(ids, mask), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += loss.item() * y.size(0)
        n += y.size(0)
    return total / max(n, 1)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total, correct, n = 0.0, 0, 0
    for batch in tqdm(loader, desc="eval", leave=False):
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        y = batch["labels"].to(device)
        logits = model(ids, mask)
        total += criterion(logits, y).item() * y.size(0)
        correct += (logits.argmax(-1) == y).sum().item()
        n += y.size(0)
    return total / max(n, 1), correct / max(n, 1)


def train_model(model, train_loader, val_loader, device,
                epochs=30, lr_c=1e-3, lr_q=5e-4, patience=5, seed=42):
    torch.manual_seed(seed)
    c_params, q_params = separate_params(model)
    groups = [{"params": c_params, "lr": lr_c, "weight_decay": 0.0}]
    if q_params:
        groups.append({"params": q_params, "lr": lr_q, "weight_decay": 1e-4})
    opt = Adam(groups)
    criterion = nn.CrossEntropyLoss()
    best_acc, best_state, wait = 0.0, None, 0
    history = {"train_loss": [], "val_loss": [], "val_acc": []}

    for ep in range(1, epochs + 1):
        t0 = time.time()
        tr_loss = train_epoch(model, train_loader, opt, criterion, device)
        va_loss, va_acc = evaluate(model, val_loader, criterion, device)
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)
        print(f"Epoch {ep:02d}/{epochs}  train={tr_loss:.4f}  val={va_loss:.4f}  "
              f"acc={va_acc*100:.2f}%  ({time.time()-t0:.1f}s)")
        if va_acc > best_acc:
            best_acc, best_state, wait = va_acc, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping at epoch {ep}")
                break
    if best_state:
        model.load_state_dict(best_state)
    return {"best_val_acc": best_acc, "history": history}

print("Training utilities ready ✓")


Training utilities ready ✓


## 7. Smoke test (tiny model, synthetic data)

Verifies forward + backward pass for all three attention types.


In [7]:
def smoke_test(attn_type, n_qubits=2, seq_len=4, q_layers=1):
    print(f"\n--- {attn_type} | qubits={n_qubits} | seq={seq_len} ---")
    model = HQCT(vocab_size=500, num_classes=4, d_model=32, n_layers=1, n_heads=2,
                 d_ff=64, max_seq_len=seq_len, attn_type=attn_type,
                 n_qubits=n_qubits, q_layers=q_layers).to(DEVICE)
    x = torch.randint(1, 500, (1, seq_len), device=DEVICE)
    mask = torch.ones(1, seq_len, dtype=torch.long, device=DEVICE)
    logits = model(x, mask)
    logits.sum().backward()
    n = sum(p.numel() for p in model.parameters())
    print(f"Logits: {logits.detach().cpu().numpy().round(3)}")
    print(f"Params: {n:,}  |  grads flow: {any(p.grad is not None for p in model.parameters())}")

smoke_test("classical", seq_len=8)
smoke_test("hqct-p", n_qubits=2, seq_len=4)
smoke_test("hqct-k", n_qubits=2, seq_len=3)   # O(N²) circuits – keep short
print("\n✓ Smoke tests passed")



--- classical | qubits=2 | seq=8 ---
Logits: [[-1.012  0.573 -0.924 -0.334]]
Params: 24,996  |  grads flow: True

--- hqct-p | qubits=2 | seq=4 ---
Logits: [[ 1.279  0.713 -1.63  -0.191]]
Params: 23,092  |  grads flow: True

--- hqct-k | qubits=2 | seq=3 ---
Logits: [[ 1.106  0.923 -0.719 -0.8  ]]
Params: 24,910  |  grads flow: True

✓ Smoke tests passed


## 8. Run a full experiment

Change the config below to match paper settings or ablations.

> **Note:** HQCT-K evaluates a quantum circuit for every query–key pair ($O(N^2)$).  
> For a quick run use `attn_type="classical"` or `"hqct-p"`, or reduce `max_seq_len` / `n_qubits`.  
> Full AG News HQCT-K takes ~3.5 h on an A100 (paper).


In [8]:
# ============ EXPERIMENT CONFIG ============
CONFIG = {
    "dataset": "ag_news",       # ag_news | imdb | sst2 | multinli
    "attn_type": "classical",   # classical | hqct-k | hqct-p
    "d_model": 64,              # paper: 256  (use 64 for quick demo)
    "n_layers": 2,              # paper: 4
    "n_heads": 2,               # paper: 4
    "d_ff": 128,                # paper: 512
    "n_qubits": 4,              # paper: 6
    "q_layers": 2,              # paper: 3
    "noise": 0.0,
    "batch_size": 16,
    "epochs": 5,                # paper: 30
    "few_shot": None,           # set to 16 for few-shot table
    "patience": 3,
}
# ===========================================

print("Loading data …")
train_ds, val_ds, test_ds, n_classes, vocab_size, max_len = load_benchmark(
    CONFIG["dataset"], seed=SEED
)
if CONFIG["few_shot"]:
    train_ds = make_few_shot(train_ds, n_classes, k=CONFIG["few_shot"], seed=SEED)
    print(f"Few-shot: {len(train_ds)} examples")

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)
print(f"{CONFIG['dataset']} | classes={n_classes} | vocab={vocab_size} | seq={max_len}")
print(f"Train={len(train_ds)}  Val={len(val_ds)}  Test={len(test_ds)}")


Loading data …


HfUriError: Invalid HF URI 'hf://datasets/ag_news@eb185aade064a813bc0b7f42de02595523103ca4/.huggingface.yaml'. Repository id must be 'namespace/name', got 'ag_news'.

In [ ]:
model = HQCT(
    vocab_size=vocab_size,
    num_classes=n_classes,
    d_model=CONFIG["d_model"],
    n_layers=CONFIG["n_layers"],
    n_heads=CONFIG["n_heads"],
    d_ff=CONFIG["d_ff"],
    max_seq_len=max_len,
    attn_type=CONFIG["attn_type"],
    n_qubits=CONFIG["n_qubits"],
    q_layers=CONFIG["q_layers"],
    noise=CONFIG["noise"],
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {CONFIG['attn_type']}  |  params: {n_params:,}")
print(f"Qubits={CONFIG['n_qubits']}  depth={CONFIG['q_layers']}  noise={CONFIG['noise']}")


In [ ]:
result = train_model(
    model, train_loader, val_loader, DEVICE,
    epochs=CONFIG["epochs"],
    patience=CONFIG["patience"],
    seed=SEED,
)

criterion = nn.CrossEntropyLoss()
test_loss, test_acc = evaluate(model, test_loader, criterion, DEVICE)
print(f"\n=== Test accuracy: {test_acc*100:.2f}% ===")
print(f"Best val accuracy: {result['best_val_acc']*100:.2f}%")


## 9. Visualise training curves & attention

In [ ]:
hist = result["history"]
epochs = range(1, len(hist["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs, hist["train_loss"], "o-", label="Train", color="#2563eb")
axes[0].plot(epochs, hist["val_loss"], "s-", label="Val", color="#dc2626")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[0].set_title("Loss")

axes[1].plot(epochs, [a * 100 for a in hist["val_acc"]], "o-", color="#059669")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy (%)")
axes[1].set_title("Validation Accuracy"); axes[1].grid(True, alpha=0.3)

fig.suptitle(f"{CONFIG['attn_type']} on {CONFIG['dataset']}", fontweight="bold")
fig.tight_layout()
plt.show()


In [ ]:
# Attention heatmap on a sample (works best with classical / hqct-p; hqct-k is slow)
model.eval()
sample = next(iter(val_loader))
ids = sample["input_ids"][:1].to(DEVICE)
mask = sample["attention_mask"][:1].to(DEVICE)

with torch.no_grad():
    logits, attns = model(ids, mask, return_attn=True)

# Average over heads, first layer
attn = attns[0][0].mean(0).cpu().numpy()  # (N, N)
# Restrict to non-padding tokens
seq_len = int(mask[0].sum().item())
attn = attn[:seq_len, :seq_len]

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(attn, cmap="viridis", aspect="auto")
ax.set_title(f"Attention heatmap ({CONFIG['attn_type']}, layer 0)")
ax.set_xlabel("Key"); ax.set_ylabel("Query")
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()
print(f"Predicted class: {logits.argmax(-1).item()}  |  True: {sample['labels'][0].item()}")


## 10. Paper reference results

| Model | AG News | IMDB | SST-2 | MultiNLI | Params |
|-------|---------|------|-------|----------|--------|
| Classical Transformer | 91.2 | 88.4 | 89.1 | 72.3 | 3.85 M |
| Linear Attention | 90.1 | 87.2 | 87.8 | 70.9 | 3.42 M |
| **HQCT-K** | **91.8** | **88.9** | 88.7 | **73.1** | **3.31 M** |
| HQCT-P | 90.9 | 88.1 | 88.3 | 71.8 | 3.28 M |

**Few-shot (16/class):** HQCT-K gains +4–5 pts over classical.


In [ ]:
# Reproduce paper-scale config (commented – long runtime)
# CONFIG_PAPER = {
#     "dataset": "ag_news",
#     "attn_type": "hqct-k",
#     "d_model": 256, "n_layers": 4, "n_heads": 4, "d_ff": 512,
#     "n_qubits": 6, "q_layers": 3, "noise": 0.0,
#     "batch_size": 32, "epochs": 30, "few_shot": None, "patience": 5,
# }
print("Notebook complete. Change CONFIG and re-run section 8 for other experiments.")
